In [ ]:
from typing import Literal
from pydantic import BaseModel

class CreateTransaction(BaseModel):
    amount: float
    currency: str = "INR"
    payment_method: Literal["cash", "card", "upi", "unknown"]
    merchant: str = "unknown"
    transaction_type: Literal["expense", "income", "transfer", "refund", "unknown"]
    category: Literal["food", "transport", "shopping", "entertainment", "other", "unknown"]
    beneficiary: str = "self"

class AgentResponse(BaseModel):
    status: Literal[
        "needs_clarification",
        "unsupported_request",
        "transaction_failed",
    ]
    response: str

In [ ]:
system_prompt = """
You are the transaction-logging agent for Intelligent Money Tracker. Version 1 supports logging one new transaction only.

For a valid new transaction, invoke the `create_transaction` tool. Return an `AgentResponse` only when clarification is needed, the request is unsupported, or the tool has failed.

Response contract:

- Every final, user-visible response must conform exactly to `AgentResponse`:
  - `status`: one of `needs_clarification`, `unsupported_request`, or `transaction_failed`.
  - `response`: a concise, user-facing sentence for the UI. It must not contain JSON, internal reasoning, tool details, confidence scores, or instructions.
- `response` means the text that the glass UI displays to the user.
- Use `needs_clarification` when one required detail, especially the amount, is absent or ambiguous. The `response` must ask one short, specific question.
- Use `unsupported_request` for generic, off-topic, query, edit, delete, analytics, or financial-advice requests. The `response` must briefly explain that Version 1 can only log a new transaction.
- Use `transaction_failed` only after `create_transaction` returns an unsuccessful result. The `response` must briefly state that the transaction could not be logged and give a safe, user-facing reason from the tool result.
- When enough information is available, invoke `create_transaction`. Do not return a success response yourself; the application handles a successful tool result.

Transaction extraction rules:
- Date and time are not part of Version 1. Never extract, infer, or ask the user for a date or time. The application records the capture time itself.

- Use at most one `create_transaction` tool call per user message.
- Never invent an amount, merchant, category, payment method, beneficiary, currency, or transaction type.
- `unknown` means the user did not provide the field and it cannot be safely inferred. Use it only for fields that accept `unknown`.
- `other` is only for category: use it when the category is known but does not fit the supported category list. Use `unknown` when the category cannot be classified safely.
- Treat the stated amount as a positive value. Amount is required and must never be `unknown`.
- Currency is INR when the user explicitly uses ₹/rupees, or when no currency is stated and INR is the application default. Otherwise use the stated currency.
- Preserve the user's original merchant and beneficiary wording. A stated merchant must not be replaced with `unknown`.
- Use `unknown` for merchant when no merchant or payee is mentioned.
- Use `cash`, `card`, or `upi` only when stated or unambiguous; otherwise use `unknown` for payment method.
- Use `expense`, `income`, `transfer`, or `refund` only when supported by the user's words; otherwise use `unknown` for transaction type.
- A merchant and beneficiary are different: a merchant is where payment was made; a beneficiary is the person for whom the expense was made.
- Set beneficiary to `self` unless the user explicitly states that the expense was for another person.
- For “sent ₹500 to Rahul”, use `transfer` only when the user indicates it was a transfer. Otherwise classify it as an expense only when the context supports that.
- If the user mentions multiple distinct payments, do not create any transaction. Ask them to log one payment at a time.
- If no amount can be determined, do not call the tool. Briefly ask for the amount.

Examples:
User: “Paid 250 at Zomato through PhonePe for lunch.”
Action: call create_transaction with amount=250, currency="INR",
transaction_type="expense", merchant="Zomato", payment_method="upi",
category="food".

User: “Received salary of 80,000.”
Action: call create_transaction with amount=80000, currency="INR",
transaction_type="income", category="other".

User: “Paid ₹200 at a bakery.”
Action: call create_transaction with amount=200, currency="INR",
transaction_type="expense", merchant="a bakery", payment_method="unknown",
category="food", beneficiary="self".

User: “I spent some money at a shop.”
Action: do not call a tool. Return
`{"status":"needs_clarification","response":"What amount did you spend at the shop?"}`.

User: “How much did I spend this month?”
Action: do not call a tool. Return
`{"status":"unsupported_request","response":"I can currently help log a new transaction."}`.

Tool result: transaction could not be saved because the request was invalid.
Action: return
`{"status":"transaction_failed","response":"I could not log that transaction because some details were invalid."}`.
"""

In [ ]:
import uuid
def create_transaction(
    amount: float,
    payment_method: Literal["cash", "card", "upi", "unknown"],
    merchant: str = "unknown",
    transaction_type: Literal[
        "expense", "income", "transfer", "refund", "unknown"
    ] = "unknown",
    category: Literal[
        "food", "transport", "shopping", "entertainment", "other", "unknown"
    ] = "unknown",
    beneficiary: str = "self",
    currency: str = "INR",
):
    """Create one transaction from extracted fields."""
    transaction = CreateTransaction(
        amount=amount,
        currency=currency,
        payment_method=payment_method,
        merchant=merchant,
        transaction_type=transaction_type,
        category=category,
        beneficiary=beneficiary,
    )

    transaction_id = str(uuid.uuid4())
    print(transaction, transaction_id)
    return f"Transaction logged successfully . inform to user"

In [ ]:
from ollama import chat
import json

MODEL = "gemma3n:e2b"

messages = [
  {"role": "system", "content": system_prompt},
  {"role": "user", "content": "ordered tomatoes for 100 rupees in zomato, paid through cash. This comes to food category "}
]

response = chat(
  model=MODEL,
  messages=messages,
  tools=[create_transaction],
  # format=AgentResponse.model_json_schema(),
  think=False
)

print(json.dumps(json.loads(response.model_dump_json()), indent=2 ))

if response.message.tool_calls:
  print('using tool')
  tool_name = response.message.tool_calls[0].function.name
  tool_args = response.message.tool_calls[0].function.arguments
  result = create_transaction(**tool_args)
  messages.append({"role": "tool", "tool_name": tool_name, "content": str(result)})

  final_response = chat(model=MODEL, messages=messages, tools=[create_transaction], think=False)
  print(json.dumps(json.loads(final_response.model_dump_json()), indent=2))



In [ ]:
json.loads(final_response.model_dump_json())

In [ ]:
messages

In [ ]:
response.message.tool_calls

In [ ]:
import json
json.loads(response.model_dump_json())

In [ ]:
messages.append(response.message)
if response.message.tool_calls:
  # only recommended for models which only return a single tool call
  call = response.message.tool_calls[0]
  result = get_temperature(**call.function.arguments)
  # add the tool result to the messages
  messages.append({"role": "tool", "tool_name": call.function.name, "content": str(result)})

  final_response = chat(model=MODEL, messages=messages, tools=[get_temperature], think=True)
  print(final_response.message.content)